# Agricultural Super-Resolution GAN - RunPod

**First-time setup:** Open a terminal in JupyterLab and run:
```bash
cd /workspace
git clone <your-repo-url> super-gan
cd super-gan
bash setup.sh
```
This installs dependencies and downloads the PlantVillage dataset. You only need to do this once.

## 1. Verify Setup

In [ ]:
import os
import torch

# Make sure we're in the right directory
os.chdir("/workspace/super-gan")

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Verify the dataset is ready
train_dir = "data/plantvillage/color_images/train"
assert os.path.isdir(train_dir), (
    f"Dataset not found at {train_dir}. "
    "Run 'bash setup.sh' in the terminal first."
)
num_classes = len(os.listdir(train_dir))
print(f"Dataset OK: {num_classes} classes found")

## 2. Load and Explore Data

In [ ]:
from training_utils import (
    load_class_names,
    load_plant_balanced_image_file_sample,
    build_plant_dict,
    data_summary,
    summarize_image_attributes,
)

train_dir = "data/plantvillage/color_images/train"

class_names = load_class_names(train_dir)
plant_dict = build_plant_dict(class_names)
sample_image_files, sample_labels, sample_classes = load_plant_balanced_image_file_sample(
    train_dir, plant_dict, 500
)

In [ ]:
data_summary(sample_image_files, sample_labels, class_names)
summarize_image_attributes(sample_image_files)

## 3. Create Dataset and DataLoader

In [ ]:
import dataset_utils
from torch.utils.data import DataLoader

dataset = dataset_utils.SuperResolutionDataset(image_files=sample_image_files)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)

print(f"Dataset: {len(dataset)} images")
print(f"Batches per epoch: {len(dataloader)}")

## 4. Import Models

In [ ]:
from generators import SimpleGenerator, ResidualGenerator, AttentionMedicalGenerator
from agricultural_sr import AgriculturalAttentionGenerator, AgriculturalDiscriminator, AgriculturalTrainer
from training_classes import AdaptiveTrainer, MedicalTrainer, EnhancedTrainer
from discriminators import SimpleDiscriminator, SpectralNormDiscriminator
import training_loop

print("All models imported successfully.")

## 5. Train

In [ ]:
generator, discriminator = training_loop.enhanced_training_loop(
    dataset,
    dataloader,
    AgriculturalTrainer,
    AgriculturalAttentionGenerator,
    AgriculturalDiscriminator,
    model_name="ag_gen_and_dis_model_500_per_class_sample_pv",
    num_epochs_per_block=2,
    total_epochs=10,
    checkpoint_dir="checkpoints",
    use_wandb=False,
)

## 6. Save Final Models

In [ ]:
import torch

os.makedirs("checkpoints", exist_ok=True)
torch.save(generator.state_dict(), "checkpoints/ag_generator.pth")
torch.save(discriminator.state_dict(), "checkpoints/ag_discriminator.pth")
print("Models saved to checkpoints/")